# Jyupiter Training Notebook

This notebook demonstrates how to train the `SvaraTransformer` model in the Raaga.AI repository using the existing training pipeline. It loads the dataset, builds the model, trains it, evaluates validation loss, and saves a checkpoint.

In [ ]:
# ── Colab / environment setup ────────────────────────────────────────────────
# This cell is safe to run anywhere. In Google Colab it clones the repo,
# installs Python dependencies, and changes the working directory so that
# all subsequent cells can find `src/`, `data/`, etc.

import importlib
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')

if IN_COLAB:
    REPO_URL = 'https://github.com/Pariseashishbannu/raagaai.git'
    REPO_DIR = Path('/content/raagaai')

    if not REPO_DIR.exists():
        print('Cloning repository...')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])
    else:
        print('Repository already cloned, pulling latest changes...')
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])

    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

    # Install Python requirements
    req_file = REPO_DIR / 'requirements.txt'
    if req_file.exists():
        print('Installing requirements...')
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_file)
        ])
    else:
        # Minimal deps needed by the notebook
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q',
            'torch', 'requests', 'beautifulsoup4',
        ])

    print(f'Working directory set to: {os.getcwd()}')
else:
    print(f'Not in Colab — running from: {os.getcwd()}')


In [ ]:
import math
import time
import sys
from pathlib import Path

import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Locate the repository root using a repo-specific marker file.
# Checking bare src/ is not reliable — Linux has a system /src directory.
def _find_repo_root() -> Path:
    marker = Path('src') / 'ml' / 'model.py'   # unique to this repo
    cwd = Path.cwd()
    # Check cwd itself (Colab: os.chdir'd into the repo in the setup cell)
    if (cwd / marker).exists():
        return cwd
    # Walk upward (local dev: notebook opened from inside the repo)
    candidate = cwd.parent
    while candidate != candidate.parent:
        if (candidate / marker).exists():
            return candidate
        candidate = candidate.parent
    raise RuntimeError(
        f'Cannot find repo root (looked for {marker}). '
        f'cwd={cwd}. In Colab, make sure the setup cell ran successfully.'
    )

repo_root = _find_repo_root()

sys.path.insert(0, str(repo_root))

from src.ml.data.dataset import SvaraDataset
from src.ml.model import SvaraTransformer
from src.ml.tokenizer import SvaraTokenizer

pretrained_checkpoint_path = repo_root / 'data' / 'models' / 'svara_transformer.pt'
notebook_checkpoint_path = repo_root / 'data' / 'models' / 'svara_transformer_notebook.pt'

print('Repository root:', repo_root)
print('Pretrained checkpoint:', pretrained_checkpoint_path)
print('Notebook checkpoint:', notebook_checkpoint_path)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', torch.cuda.get_device_properties(0).total_memory / 1e9)


: 

## Build Dataset

Collect svara sequences from multiple sources and write them to
`data/training/svara_sequences.jsonl` so the training cells below can load them.

**Sources (in priority order):**

| # | Source | Notes |
|---|--------|-------|
| 1 | **Synthetic** — local raga definitions | Always enabled; no network needed |
| 2 | **karnatik.com** | Web scrape; toggle `SCRAPE_KARNATIK` |
| 3 | **shivkumar.org** | Web scrape; toggle `SCRAPE_SHIVKUMAR` |
| 4 | **Saraga dataset** | Set `SARAGA_DIR` to the extracted dataset path |
| 5 | **CompMusic Dunya API** | Set `DUNYA_TOKEN` (free registration required) |

Skip this section if `data/training/svara_sequences.jsonl` already exists and you
do not want to rebuild it.


In [ ]:
# Install web-scraping deps if missing (safe to run multiple times)
import importlib
import subprocess
import sys


def _ensure(pkg, import_name=None):
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])


_ensure('requests')
_ensure('beautifulsoup4', 'bs4')
print('Dependencies ready.')


### Configuration

Edit the variables below to control which data sources are used.


In [ ]:
import os

# ── Data source toggles ──────────────────────────────────────────────────
SCRAPE_KARNATIK = True   # scrape karnatik.com (respectful rate limiting built in)
SCRAPE_SHIVKUMAR = True  # scrape shivkumar.org
MAX_WEB_PAGES = 50       # max pages per web source

SARAGA_DIR = None        # e.g. r'C:/datasets/saraga-carnatic-1.5' or None to skip

DUNYA_TOKEN = os.environ.get('DUNYA_TOKEN', '')  # leave blank to skip
DUNYA_MAX_REC = 100      # max recordings to fetch from Dunya

# ── Synthesis / augmentation ─────────────────────────────────────────────
SYNTHETIC_PER_RAGA = 200  # synthetic sequences generated per raga
AUGMENT = True            # apply sub-phrase repetition + window sampling

# ── Output ───────────────────────────────────────────────────────────────
OUTPUT_PATH = repo_root / 'data' / 'training' / 'svara_sequences.jsonl'
FORCE_REBUILD = False     # set True to overwrite existing dataset

print('Config loaded.')
print(f'  karnatik.com scrape : {SCRAPE_KARNATIK} (max {MAX_WEB_PAGES} pages)')
print(f'  shivkumar.org scrape: {SCRAPE_SHIVKUMAR} (max {MAX_WEB_PAGES} pages)')
print(f'  Saraga dir          : {SARAGA_DIR}')
print(f"  Dunya token         : {'set' if DUNYA_TOKEN else 'not set'}")
print(f'  Synthetic/raga      : {SYNTHETIC_PER_RAGA}')
print(f'  Augment             : {AUGMENT}')
print(f'  Output              : {OUTPUT_PATH}')
print(f'  Force rebuild       : {FORCE_REBUILD}')


### Run scrapers and save JSONL

This cell collects sequences from all enabled sources, applies augmentation,
shuffles, and writes the final dataset file.


In [ ]:
import json as _json
import logging
import random
from collections import Counter

from src.ml.data.scraper import (
    DunyaFetcher,
    KarnatikScraper,
    LocalRagaSynthesizer,
    SaragaScraper,
    ShivkumarScraper,
    augment_sequences,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

if OUTPUT_PATH.exists() and not FORCE_REBUILD:
    print(f'Dataset already exists at {OUTPUT_PATH} — skipping build.')
    print('Set FORCE_REBUILD = True in the config cell to regenerate it.')
else:
    all_sequences: list[dict] = []

    # 1. Synthetic sequences (always)
    print('[1/5] Generating synthetic sequences from local raga definitions...')
    synth = LocalRagaSynthesizer(str(repo_root / 'data' / 'ragas'))
    synthetic = synth.synthesize(sequences_per_raga=SYNTHETIC_PER_RAGA)
    all_sequences.extend(synthetic)
    print(f'      {len(synthetic):,} synthetic sequences')

    # 2. karnatik.com
    if SCRAPE_KARNATIK:
        print(f'[2/5] Scraping karnatik.com (max {MAX_WEB_PAGES} pages)...')
        karnatik_seqs = KarnatikScraper().scrape(max_pages=MAX_WEB_PAGES)
        all_sequences.extend(karnatik_seqs)
        print(f'      {len(karnatik_seqs):,} sequences from karnatik.com')
    else:
        print('[2/5] karnatik.com skipped (SCRAPE_KARNATIK=False)')

    # 3. shivkumar.org
    if SCRAPE_SHIVKUMAR:
        print(f'[3/5] Scraping shivkumar.org (max {MAX_WEB_PAGES} pages)...')
        shiv_seqs = ShivkumarScraper().scrape(max_pages=MAX_WEB_PAGES)
        all_sequences.extend(shiv_seqs)
        print(f'      {len(shiv_seqs):,} sequences from shivkumar.org')
    else:
        print('[3/5] shivkumar.org skipped (SCRAPE_SHIVKUMAR=False)')

    # 4. Saraga dataset
    if SARAGA_DIR:
        print(f'[4/5] Parsing Saraga dataset from {SARAGA_DIR}...')
        saraga_seqs = SaragaScraper().scrape(SARAGA_DIR)
        all_sequences.extend(saraga_seqs)
        print(f'      {len(saraga_seqs):,} sequences from Saraga')
    else:
        print('[4/5] Saraga skipped (SARAGA_DIR not set)')

    # 5. CompMusic Dunya API
    if DUNYA_TOKEN:
        print(f'[5/5] Fetching from Dunya (max {DUNYA_MAX_REC} recordings)...')
        dunya_seqs = DunyaFetcher(DUNYA_TOKEN).fetch(max_recordings=DUNYA_MAX_REC)
        all_sequences.extend(dunya_seqs)
        print(f'      {len(dunya_seqs):,} sequences from Dunya')
    else:
        print('[5/5] Dunya skipped (DUNYA_TOKEN not set)')

    if not all_sequences:
        raise RuntimeError('No sequences collected — check your data source configuration.')

    # Augment
    if AUGMENT:
        print(f'\nAugmenting {len(all_sequences):,} sequences...')
        all_sequences = augment_sequences(all_sequences)
        print(f'  -> {len(all_sequences):,} sequences after augmentation')

    # Shuffle and save
    random.shuffle(all_sequences)
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(OUTPUT_PATH, 'w', encoding='utf-8') as _f:
        for seq in all_sequences:
            _f.write(_json.dumps(seq, ensure_ascii=False) + '\n')

    # Stats
    raga_counts = Counter(s['raga_id'] for s in all_sequences)
    section_counts = Counter(s['section'] for s in all_sequences)
    total_svaras = sum(len(s['svaras']) for s in all_sequences)

    print(f'\n=== Dataset saved to {OUTPUT_PATH} ===')
    print(f'  Total sequences : {len(all_sequences):,}')
    print(f'  Unique ragas    : {len(raga_counts)}')
    print(f'  Sections        : {dict(section_counts)}')
    print(f'  Total svaras    : {total_svaras:,}')
    print(f'  Avg seq length  : {total_svaras / len(all_sequences):.1f}')
    print(f'\nTop 10 ragas by sequence count:')
    for raga, cnt in raga_counts.most_common(10):
        print(f'    {raga:<30} {cnt:,}')


## Load Dataset

The Raaga.AI training data is expected at `data/training/svara_sequences.jsonl`. This notebook loads that file and inspects the dataset size.

In [ ]:
data_path = repo_root / 'data' / 'training' / 'svara_sequences.jsonl'
print('Training data path:', data_path)
print('Exists:', data_path.exists())

if not data_path.exists():
    raise FileNotFoundError(
        f'Training data not found at {data_path}\n'
        'Run the \'Build Dataset\' section at the top of this notebook first, '
        'then re-run this cell.'
    )

tokenizer = SvaraTokenizer()
dataset = SvaraDataset(str(data_path), tokenizer, max_length=256)
print('Dataset size:', len(dataset))
print('Vocabulary size:', tokenizer.vocab_size)

sample = dataset[0] if len(dataset) > 0 else None
print('Sample keys:', list(sample.keys()) if sample else None)
if sample:
    print('Input IDs length:', sample['input_ids'].shape)
    print('Labels length:', sample['labels'].shape)


## Preprocess Data

Create train/validation splits and PyTorch data loaders for batch training. This step uses the same sequence format and tokenizer as the repo's training script.

In [ ]:
batch_size = 32
val_split = 0.1
max_seq_len = 256

val_size = max(1, int(len(dataset) * val_split))
train_size = len(dataset) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == 'cuda'),
)
val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == 'cuda'),
)

print('Train samples:', len(train_dataset))
print('Validation samples:', len(val_dataset))
print('Batch size:', batch_size)


## Define Model Architecture

Instantiate the `SvaraTransformer` from `src/ml/model.py` with the same architecture used by the existing training script.

In [ ]:
model = SvaraTransformer(
    vocab_size=tokenizer.vocab_size,
    embed_dim=128,
    num_heads=4,
    num_layers=4,
    max_seq_len=max_seq_len,
    dropout=0.1,
).to(device)


def count_parameters(module: torch.nn.Module) -> int:
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

print('Model parameter count:', count_parameters(model))
print(model)


## Compile Model

Set up optimizer, scheduler, and training hyperparameters. This notebook mirrors the repo's `scripts/train_model.py` defaults.

If a pretrained checkpoint exists, it is loaded and training is skipped; otherwise the notebook will train for the full number of epochs on the available dataset.


In [ ]:
learning_rate = 3e-4
grad_clip = 1.0
full_epochs = 50

# If a pretrained checkpoint exists, use it and skip full training.
# Otherwise, train from scratch on the available dataset.
if notebook_checkpoint_path.exists():
    checkpoint_to_load = notebook_checkpoint_path
elif pretrained_checkpoint_path.exists():
    checkpoint_to_load = pretrained_checkpoint_path
else:
    checkpoint_to_load = None

if checkpoint_to_load is not None:
    epochs = 0
    print('Loading pretrained checkpoint, skipping training:', checkpoint_to_load)
    checkpoint = torch.load(checkpoint_to_load, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f'Loaded pretrained model weights from: {checkpoint_to_load}')
else:
    epochs = full_epochs
    print('No pretrained checkpoint found; training for full epochs:', epochs)

optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=max(1, epochs), eta_min=learning_rate * 0.1)

print('Learning rate:', learning_rate)
print('Epochs:', epochs)
print('Checkpoint to load:', checkpoint_to_load)


## Train Model

Run the training loop. The notebook logs training and validation loss per epoch and saves the best checkpoint.

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    steps = 0
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        output = model(input_ids, labels=labels)
        loss = output['loss']
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        total_loss += loss.item()
        steps += 1
    return total_loss / max(steps, 1)


def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    steps = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            output = model(input_ids, labels=labels)
            total_loss += output['loss'].item()
            steps += 1
    return total_loss / max(steps, 1)

output_dir = repo_root / 'data' / 'models'
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = notebook_checkpoint_path

best_val_loss = float('inf')
patience = 10
patience_counter = 0
start_time = time.time()

if epochs == 0:
    print('Pretrained checkpoint loaded; skipping training loop.')
else:
    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_loss = evaluate(model, val_loader, device)
        scheduler.step()
        elapsed = time.time() - start_time
        print(f'Epoch {epoch}/{epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f} - lr: {scheduler.get_last_lr()[0]:.2e} - elapsed: {elapsed:.0f}s')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'config': {
                    'vocab_size': tokenizer.vocab_size,
                    'embed_dim': 128,
                    'num_heads': 4,
                    'num_layers': 4,
                    'max_seq_len': max_seq_len,
                    'dropout': 0.0,
                },
                'tokenizer_vocab': tokenizer.token_to_id,
                'epoch': epoch,
                'val_loss': best_val_loss,
            }, checkpoint_path)
            print(f'  → Saved best model to {checkpoint_path}')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping after {patience} epochs without improvement.')
                break

    print('Training complete. Best validation loss:', best_val_loss)


## Evaluate Model

Run a final evaluation on the held-out validation split to confirm the checkpoint quality.

In [ ]:
final_val_loss = evaluate(model, val_loader, device)
print('Final validation loss:', final_val_loss)


## Save Trained Model

The notebook saves the trained checkpoint and generates a sample melody using the same tokenizer.

In [ ]:
print('Checkpoint saved at:', checkpoint_path)

model.eval()
prefix_tokens = tokenizer.encode_conditioning('shankarabharanam', 'adi', 'pallavi', 'madhyama')
prefix_tensor = torch.tensor([prefix_tokens], device=device)
generated = model.generate(prefix_tensor, max_new_tokens=32, temperature=0.8, top_k=15)
decoded = tokenizer.decode(generated[0].cpu().tolist())
print('Generated sequence:')
print(' '.join(decoded))
